In [0]:
# =====================================
# NOTEBOOK : NB_SILVER_PAYMENT
# LAYER    : SILVER
# TARGET   : ODS_PAYMENT
# =====================================

In [0]:
%run ../COMMON/NB_CONFIG

In [0]:
%run ../COMMON/NB_CONNECTION

In [0]:
%run ../COMMON/NB_UTILS

In [0]:
payment_df = spark.read.jdbc(
    url=srcJdbcUrl,
    table="BRONZE.PAYMENT",
    properties=connectionProperties
)

print("PAYMENT COUNT :", payment_df.count())

In [0]:
null_count = payment_df.filter(F.col("PAYMENT_ID").isNull()).count()
if null_count > 0:
    raise Exception(f"NULL PAYMENT_ID FOUND : {null_count}")

In [0]:
ods_payment_df = payment_df.select(
    "PAYMENT_ID","CLAIM_ID","PAYMENT_DATE","PAYMENT_AMOUNT",
    "PAYMENT_STATUS","SRC_CREATED_DATETIME"
)

In [0]:
ods_payment_df = (
    ods_payment_df
    .withColumn("PAYMENT_YEAR", F.year("PAYMENT_DATE"))
    .withColumn("PAYMENT_MONTH", F.month("PAYMENT_DATE"))
    .withColumn("PAYMENT_DELAY_DAYS", F.datediff("PAYMENT_DATE","SRC_CREATED_DATETIME"))
    .withColumn("STATUS_FLAG",
        F.when(F.col("PAYMENT_STATUS") == "PAID","CLOSED")
         .when(F.col("PAYMENT_STATUS") == "PENDING","OPEN")
         .otherwise("UNKNOWN")
    )
)

In [0]:
ods_payment_df = ods_payment_df.fillna({
    "PAYMENT_STATUS":"UNKNOWN",
    "PAYMENT_AMOUNT":0
})

In [0]:
window_spec = Window.partitionBy("PAYMENT_ID").orderBy(F.col("SRC_CREATED_DATETIME").desc())

ods_payment_df = (
    ods_payment_df
    .withColumn("RN", F.row_number().over(window_spec))
    .filter(F.col("RN") == 1)
    .drop("RN")
)

In [0]:
ods_payment_df = ods_payment_df.withColumn(
    "HASH_VALUE",
    F.sha2(
        F.concat_ws("|",
            "CLAIM_ID","PAYMENT_DATE","PAYMENT_AMOUNT","PAYMENT_STATUS"
        ),256
    )
)

In [0]:
ods_payment_df = (
    ods_payment_df
    .withColumn("LOAD_DATE", F.current_timestamp())
    .withColumn("IS_CURRENT", F.lit("Y"))
    .withColumn("EFFECTIVE_DATE", F.current_timestamp())
    .withColumn("END_DATE", F.lit("9999-12-31").cast("timestamp"))
)

In [0]:
target_count = spark.table(ods_payment_table).count()
print(f"TARGET RECORD COUNT : {target_count}")

if target_count == 0:
    # Initial full load
    ods_payment_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(ods_payment_table)

    print("INITIAL LOAD COMPLETED")
    dbutils.notebook.exit("Initial load done")

else:
    print("SCD2 LOAD STARTED")
    target_df = spark.table(ods_payment_table)
    current_df = target_df.filter(F.col("IS_CURRENT") == "Y")
    print(f"CURRENT ACTIVE RECORDS : {current_df.count()}")

In [0]:
changed_df = (
    ods_payment_df.alias("s")
    .join(current_df.alias("t"), "PAYMENT_ID")
    .filter(F.col("s.HASH_VALUE") != F.col("t.HASH_VALUE"))
    .select("s.*")
)

In [0]:
new_payment_df = (
    ods_payment_df.alias("s")
    .join(current_df.alias("t"), "PAYMENT_ID", "left_anti")
)

In [0]:
delta_table = DeltaTable.forName(spark, ods_payment_table)

expire_df = changed_df.select("PAYMENT_ID").distinct()

delta_table.alias("t").merge(
    expire_df.alias("s"),
    "t.PAYMENT_ID = s.PAYMENT_ID AND t.IS_CURRENT = 'Y'"
).whenMatchedUpdate(
    set={
        "IS_CURRENT": "'N'",
        "END_DATE": "current_timestamp()"
    }
).execute()

In [0]:
changed_insert_df = (
    changed_df
    .withColumn("LOAD_DATE", F.current_timestamp())
    .withColumn("IS_CURRENT", F.lit("Y"))
    .withColumn("EFFECTIVE_DATE", F.current_timestamp())
    .withColumn("END_DATE", F.lit("9999-12-31").cast("timestamp"))
)

changed_insert_df.write.format("delta").mode("append").saveAsTable(ods_payment_table)

In [0]:
new_payment_df = (
    new_payment_df
    .withColumn("LOAD_DATE", F.current_timestamp())
    .withColumn("IS_CURRENT", F.lit("Y"))
    .withColumn("EFFECTIVE_DATE", F.current_timestamp())
    .withColumn("END_DATE", F.lit("9999-12-31").cast("timestamp"))
)

new_payment_df.write.format("delta").mode("append").saveAsTable(ods_payment_table)

In [0]:
display(
    spark.sql(f"""
    SELECT *
    FROM {ods_payment_table}
    ORDER BY PAYMENT_ID
    """)
)